# Experiment 1: Budget-Accuracy Curves (Search vs. Verify)

**Learning objective (course Goal 2):** plot your model's budget-accuracy data (search vs. verify at equal budget), then read the full curve shape from the cached family (the six-model p-spectrum) and see how voting effectiveness changes with p. Here p = pass@1, the fraction of problems a model solves correctly in one generation attempt (values in the Section 1 table); voting is guaranteed to work only for p>1/2. The 8-config preset gives the full curve; the 3-core/2-config fallbacks give the equal-budget search-vs-verify comparison only (preset details in Sections 1-2; Qwen3-4B runs the 3-core configs by default).

**The question:** with a fixed budget of **50 reasoning attempts** (N x (1+M) = 50, plus one low-budget verify point at 10 calls), is it better to *search* (generate N candidates and take a majority vote) or *verify* (generate N candidates, have the model score each candidate M times, and return the highest-scored one; ties broken by the candidate that appears first)?

> **Note on scope:** this course's claim is a *conditional statement*; under a **weak verifier (LLM self-scoring) and a moderate budget (B = 50)** on the MATH-500 subset, search wins.
>
> <details><summary>Our measured results (check after you run your own experiment)</summary>
>
> We ran the headline comparison (our main 3-seed run) on 100 problems x 3 seeds: search N=50 beat verify N=5/M=9 by +7.3 to +10.0 percentage points (pp) across our three main models (Gemma-3-1B, Gemma-3-4B, Qwen3-4B; 95% CIs exclude 0). Compare your own numbers with ours, and remember the small-sample caution below: the difference between two noisy estimates is noisier than either one alone.
>
> </details>
>
> Small-sample caution: with 6 problems, accuracy estimates carry roughly ±20 pp at one standard error (about ±40 pp at 95% confidence). Treat your numbers as directional, not precise.


---

## 1. Choose Your Model (Collaborative p-Spectrum)

Pick **one** model that fits your device (and your interest). Different students pick different models, and the class pools its curves into a **class p-spectrum**.

| Tier | Model | p (pass@1 on the 100-problem seed-0 subset) | GPU memory (Q8_0, incl. KV) |
|------|-------|:---:|:---:|
| Low | Gemma-3-1B | 0.37 | ≈1.3 GB |
| Low | Qwen3-0.6B | 0.46 | ≈0.8 GB |
| Low | Phi-4-Mini | 0.54 | ≈5.5 GB |
| Mid | Gemma-3-4B | 0.61 | ≈5.3 GB |
| Mid | Qwen3-1.7B | 0.69 | ≈2.2 GB |
| High | Qwen3-4B | 0.73 | ≈5.3 GB |

*Tier labels roughly track p, refined by device fit (memory + throughput) rather than a strict size ranking: the two 4B models share ≈5.3 GB, but Gemma-3-4B's full 8-config grid is a long run (≈6-7 h; capability is not just size). Memory figures are model file + KV cache; actual usage adds a couple of GB of process overhead. p values come from single pass@1 runs (one generation per problem) on the same 100-problem seed-0 subset, with the same Q8_0 quantization and sampling settings as the cached curves (≈±5 pp at one standard error on 100 problems).*

**p-spectrum placement:** voting has a mathematical guarantee when p>1/2 (Chernoff bound: majority-vote accuracy approaches 1 exponentially fast in N; this assumes approximately independent samples, but temperature sampling from one model is correlated, so treat the bound as the idealized case). For p<=1/2 there is **no guarantee**: in our spectrum only the p=0.62 and p=0.70 models have it. Voting may still help if wrong answers are dispersed (in the 3-seed run, our p=0.37 model's search beat its verify by +7.3 pp at equal budget) or hurt if errors are concentrated. Your results are evidence about the boundary of the p>1/2 sufficient condition, not a refutation of the main claim.

**Time estimates (approximate; one config at a time on a laptop GPU):** time = calls x ≈700 tokens / your measured throughput. Qwen3-4B runs the 3-core configs instead (N=50/M=0, N=5/M=9, N=25/M=1, all at budget 50): it gives the equal-budget comparison, but no multi-budget curve. Use the cached family for the curve shape, or the low-throughput fallback below.

| Model | 8 configs x 6 problems (1,470 calls) | 3-core configs x 6 problems (900 calls) |
|-------|:---:|:---:|
| Gemma-3-1B | ≈2-3 h | ≈1-2 h |
| Qwen3-0.6B | ≈1-2 h | ≈1-2 h |
| Qwen3-1.7B | ≈3-4 h | ≈1-2 h |
| Phi-4-Mini | ≈7-8 h (consider the 3-core fallback) | ≈4-5 h |
| Gemma-3-4B | ≈6-7 h (consider the 3-core fallback) | ≈3-4 h |
| Qwen3-4B | - (runs the 3-core configs by default) | ≈3-4 h |

If your measured throughput is below ≈30 tok/s, reduce the config list to N=50/M=0 and N=5/M=9 (the equal-budget pair) or use the cache layer (Section 3b).

**Quantization note:** all data use GGUF Q8_0, an 8-bit near-lossless (not identical) approximation of the original weights. Absolute accuracies are therefore **not directly comparable** to unquantized runs; only directional comparisons are valid.

**Set your choices here:**


In [ ]:
# --- Student configuration -------------------------------------------------
# 1) Pick your model (one of the six below).
MODEL = "gemma-3-1b"      # gemma-3-1b | qwen3-0.6b | phi-4-mini | gemma-3-4b | qwen3-1.7b | qwen3-4b
# 2) Number of problems to run (the shared 6-problem subset = first 6 of
#    our 100-problem subset, same seed). Keep 6 for comparability.
N_PROBLEMS = 6
# NOTE: the 6-problem run is a small sample, so your curve is a TREND
# ILLUSTRATION: expect small deviations from the cached 100-problem curves;
# compare directions, not exact values.
# ---------------------------------------------------------------------------
import os, subprocess, sys, json
# Windows: make CUDA runtime DLLs findable (GPU builds of llama-cpp-python need them)
if os.name == "nt":
    _torch_lib = os.path.join(sys.prefix, "Lib", "site-packages", "torch", "lib")
    if os.path.isdir(_torch_lib):
        os.environ["PATH"] = _torch_lib + os.pathsep + os.environ.get("PATH", "")
        os.add_dll_directory(_torch_lib)
# Dataset loading: fastest path first (offline cache -> probe -> mirror)
_hf_home = os.environ.get("HF_HOME", os.path.join(os.path.expanduser("~"), ".cache", "huggingface"))
_ds_cache = os.path.join(_hf_home, "datasets")
_has_cache = False
if os.path.isdir(_ds_cache):
    for _name in os.listdir(_ds_cache):
        if "math-500" in _name.lower() or "gsm8k" in _name.lower():
            _has_cache = True
            break
if _has_cache:
    os.environ["HF_HUB_OFFLINE"] = "1"  # cache present: fully offline, instant
else:
    import socket
    try:
        socket.create_connection(("huggingface.co", 443), timeout=3).close()
    except OSError:
        os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")  # official unreachable: mirror

# Locate the repository root (walk up until scripts/run_experiment.py is found)
# and switch to it, so relative paths work no matter where Jupyter was started.
_REPO_ROOT = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_REPO_ROOT, "scripts", "run_experiment.py")):
    _parent = os.path.dirname(_REPO_ROOT)
    if _parent == _REPO_ROOT:
        raise RuntimeError("Could not locate the repository root (scripts/run_experiment.py not found).")
    _REPO_ROOT = _parent
if os.path.abspath(os.getcwd()) != _REPO_ROOT:
    os.chdir(_REPO_ROOT)
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

# The eight representative configs (layer-2 small-model path; layer-3 core subset below)
CONFIGS_LAYER2 = ["N=5,M=0", "N=10,M=0", "N=20,M=0", "N=50,M=0",
                "N=5,M=1", "N=5,M=9", "N=10,M=4", "N=25,M=1"]  # 4 search budgets + 4 verify splits
CONFIGS_LAYER3 = ["N=50,M=0", "N=5,M=9", "N=25,M=1"]   # Qwen3-4B core configs (~3-5 h)

OUT_DIR = os.path.join("data", "results", "experiment1_" + MODEL)

print("Model:", MODEL, "| problems:", N_PROBLEMS)
print("Output dir:", OUT_DIR)
print("(Model files are loaded from the model/ folder in this repository; see the README for the download list.)")


---

## 2. Run the Experiment

Run the experiment with the preset code below (this executes `run_experiment.py` from the `scripts/` folder). **Do not skip this cell if you are running on a GPU:** it generates your own data points.

> The preset runs 8 configs on the shared 6-problem subset (the first 6 of the 100 problem IDs in `data/subsets/math500_subset.json`): 4 search budgets (N=5/10/20/50) for your budget-accuracy curve, and 4 verify splits, three at budget 50 (N=5/M=9, N=10/M=4, N=25/M=1) plus one at 10 calls (N=5/M=1). That is 1,470 calls total (see the time table in Section 1). Qwen3-4B runs the 3-core configs instead (times in the same table).
>
> All cached and student runs share the same sampling settings: temperature 0.8, top-p 0.95 (generation up to 2048 tokens, judging up to 256), per-config deterministic seeds (ties broken by the first candidate, as in the question above).

> **First run downloads the dataset** (MATH-500, ≈1 MB) from the Hugging Face Hub; later runs reuse the local cache. If the first load looks slow, that is the model file loading (hundreds of MB to a few GB), not a stuck cell.
>
> <details><summary>Download trouble? (mirror workaround)</summary>
>
> The setup cell above already tries the mirror automatically when the Hub is unreachable; if a download still fails, paste this at the very top of the experiment cell (before any dataset or model import) and re-run:
>
> ```python
> import os
> os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
> ```
>
> </details>
>
> If you are on the **cache layer**, skip to Section 3b. (No GPU? You may still run the experiment on CPU with the 2-config fallback, or use the cache layer.)


In [ ]:
# Run the experiment (incremental records; safe to interrupt and resume with the same command)
configs = CONFIGS_LAYER2 if MODEL != "qwen3-4b" else CONFIGS_LAYER3
cmd = [sys.executable, "scripts/run_experiment.py",
       "--model", MODEL, "--seeds", "0",
       "--configs"] + configs + [
       "--max-problems", str(N_PROBLEMS),
       "--out-dir", OUT_DIR, "--finalize"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Experiment finished. Records saved under", OUT_DIR)

---

## 3. Load Results and Plot Your Curve

The analysis script rebuilds statistics directly from the recorded calls (the records are the raw data). To read your figure, see Section 3b.1 below.


In [ ]:
# Load / aggregate results (tolerant: no data yet -> friendly message)
exp = None
exp_file = os.path.join(OUT_DIR, "exp1.json")
if not os.path.exists(exp_file):
    try:
        from scripts.analyze_experiment import main as analyze_main
        analyze_main(["--records", OUT_DIR, "--model", MODEL, "--out", exp_file])
    except Exception as e:
        # Distinguish "no records yet" from real aggregation failures
        if not os.path.isdir(OUT_DIR):
            print("Note: no experiment data yet - run the experiment cell first, then re-run this cell.")
        else:
            print("Aggregation failed:", type(e).__name__, str(e)[:200])
if exp_file and os.path.exists(exp_file):
    with open(exp_file, encoding="utf-8") as f:
        exp = json.load(f)
    print("Loaded", len(exp["configs"]), "configs for", MODEL)

In [ ]:
# Plot the budget-accuracy curve (PPT-style: white bg, no top/right spines, thin grid, large fonts)
import matplotlib.pyplot as plt
from matplotlib import patheffects

if exp is None:
    print("Note: nothing to plot yet - run the experiment cell first.")
else:
    _PALETTE = {"gemma-3-1b": "#C0392B", "phi-4-mini": "#27AE60", "qwen3-0.6b": "#8E44AD",
                "gemma-3-4b": "#E67E22", "qwen3-1.7b": "#16A085", "qwen3-4b": "#1F4E79"}
    _color = _PALETTE.get(MODEL, "#2E75B6")
    # x = config index (several configs share the same 50-call consumption, so we
    # spread them along the x-axis and annotate the real consumption below)
    xs = list(range(len(exp["configs"])))
    ys = [c["accuracy"] for c in exp["configs"]]
    labels = ["N=%d/M=%d" % (c["config"]["N"], c["config"]["M"]) for c in exp["configs"]]
    costs = ["%d calls" % (c["config"]["N"] * (1 + c["config"]["M"])) for c in exp["configs"]]
    fig, ax = plt.subplots(figsize=(7.6, 4.6), dpi=150)
    ax.plot(xs, ys, "o-", color=_color, linewidth=2.0, markersize=6.5,
            markeredgewidth=0.8, markeredgecolor="#FFFFFF", zorder=3)
    for i, (x, y, lab, cost) in enumerate(zip(xs, ys, labels, costs)):
        # Steep segments (big drop between configs) would cross an above-label:
        # place that config's label to the right of the point instead.
        _steep = i > 0 and abs(ys[i] - ys[i - 1]) > 0.1
        if _steep:
            ax.annotate(lab + chr(10) + "(" + cost + ")", (x, y), textcoords="offset points",
                        xytext=(10, 0), fontsize=9.5, ha="left", va="center",
                        color="#333333",
                        path_effects=[patheffects.withStroke(linewidth=3, foreground="white")])
        else:
            ax.annotate(lab + chr(10) + "(" + cost + ")", (x, y), textcoords="offset points",
                        xytext=(0, 9), fontsize=9.5, ha="center", color="#333333",
                        path_effects=[patheffects.withStroke(linewidth=3, foreground="white")])
    ax.set_xticks([])
    ax.set_xlim(-0.35, len(xs) - 1 + 0.35)
    ax.set_xlabel("Configs (inference calls consumed annotated above each point)", fontsize=12)
    ax.set_ylabel("Accuracy (MATH-500 subset)", fontsize=12)
    ax.set_ylim(min(ys) - 0.04, max(ys) + 0.06)
    ax.grid(True, axis="y", color="#E3E3E3", linewidth=0.7, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#999999")
    ax.spines["bottom"].set_color("#999999")
    ax.tick_params(labelsize=11, colors="#333333")
    fig.tight_layout()
    plt.show()


---

## 3b. Reading the Curve Family (and the No-GPU Cache Layer)

If you do not have a GPU, your GPU cannot fit any of the Section 1 models, or your run is still in progress, you can still complete the analysis: we include the aggregated curves of our runs in `data/cache_subset/curves.json` (MATH-500, 100 problems, seed 0, GGUF Q8_0). The cell below loads them and plots the p-spectrum curve family; your own curve (if you ran it) is drawn in your model's color on top.

**Three kinds of data in this figure, read them differently** (cost = calls consumed; equal to the budget for in-budget points):

1. **Main curves (distinct line styles per model, shown in the legend; e.g. Qwen3-1.7B uses dash-dot):** the *search* strategy of each model at budgets 5/10/20/50, from our 100-problem runs. Statistically meaningful for across-model comparisons at the same budget (within one model, budget-to-budget differences can sit at the noise level: 1 pp = 1 problem on 100).
2. **Hollow markers + lighter dashed extension (the three main models, the same three as the headline run; these points cost 60-100 calls):** *over-budget reference points* (N=10, M=5-9) answering "what if we kept verifying past the budget?" The dashed extension is only a visual reference: it connects each search line's 50-call endpoint to that model's over-budget verify points. The over-budget points stay within ≈6 pp of each model's N=5/M=9 budget-50 verify level (e.g. Qwen3-4B: 0.74; Gemma-3-4B: 0.66; Gemma-3-1B: 0.37). The flat/fluctuating trend is itself the lesson: beyond the budget, deeper verification does not keep paying off.
3. **Your curve (model color):** your 6-problem run, search as a solid line drawn thicker than the cached curves so you can tell it apart. Your verify points are **not** overlaid here: on 6 problems they are not comparable to the cached 100-problem curves at the same budget granularity (your equal-budget search-vs-verify comparison lives in your own budget-accuracy figure, Section 3). Treat your curve as a directional illustration (noise note in the header).

> Data scope: the cached curves come from our 100-problem runs (seed 0, GGUF Q8_0; the full grid (the 8-config grid plus the over-budget reference configs) for the three main models; the 8-config student grid for the other three). The 3-seed headline numbers in the note at the top come from a separate run; the cached curves are single-seed (seed 0) illustrations. The two sets differ at the noise level, so do not compare them directly. Small-sample noise applies to your 6-problem run only.


In [ ]:
# Load the cached curve family (included with the course in data/cache_subset/)
import json

cached = {}
curves_path = os.path.join("data", "cache_subset", "curves.json")
if os.path.exists(curves_path):
    for cv in json.load(open(curves_path, encoding="utf-8"))["curves"]:
        if cv["dataset"] == "MATH-500":
            cached[cv["key"]] = cv
if cached:
    print("Cached curve families found:", sorted(cached.keys()))
else:
    print("No cached curves found - they are included with the course release (re-download the course release or check that data/cache_subset/curves.json sits next to this notebook; see Section 3b)")
print("(Cached data: 100 problems, seed 0, GGUF Q8_0 - see the data-scope note in Section 3b.)")

# Plot the p-spectrum curve family (PPT-style: triple encoding color/linestyle/marker,
# legend on top outside, white bg, no top/right spines, thin grid)
import matplotlib.pyplot as plt

_MODELS = [
    ("gemma-3-1b", "Gemma-3-1B", "dashed", "v", "#C0392B"),
    ("phi-4-mini", "Phi-4-Mini", "dotted", "*", "#27AE60"),
    ("qwen3-0.6b", "Qwen3-0.6B", "dashed", "D", "#8E44AD"),
    ("gemma-3-4b", "Gemma-3-4B", "solid", "s", "#E67E22"),
    ("qwen3-1.7b", "Qwen3-1.7B", "dashdot", "^", "#16A085"),
    ("qwen3-4b", "Qwen3-4B", "solid", "o", "#1F4E79"),
]
fig, ax = plt.subplots(figsize=(7.6, 5.6), dpi=150)
_handles = []  # explicit legend order: 6 models + your 2 curves
for key, label, ls, marker, color in _MODELS:
    cv = cached.get(key)
    if cv is None:
        continue
    # Search strategy only (cost 5/10/20/50, unique x) - line, matches the PPT figure
    search = sorted([c for c in cv["configs"] if c["config"]["strategy"] == "search"],
                    key=lambda c: c["cost"])
    if search:
        xs = [c["cost"] for c in search]
        ys = [c["accuracy"] for c in search]
        ln, = ax.plot(xs, ys, color=color, marker=marker, linestyle=ls,
                      linewidth=2.0, markersize=6.5, markeredgewidth=0.8,
                      markeredgecolor="#FFFFFF", label=label, zorder=3)
        _handles.append(ln)
    # Over-budget reference points (N=10, M=5-9, cost 60-100): hollow markers,
    # dashed extension from the search line's 50-call end (trend if budget were larger)
    over = [c for c in cv["configs"] if c["cost"] > 50]
    if search and over:
        # dashed extension line (from the search line's 50-call end; no marker at 50)
        ext_x = [search[-1]["cost"]] + [c["cost"] for c in over]
        ext_y = [search[-1]["accuracy"]] + [c["accuracy"] for c in over]
        ax.plot(ext_x, ext_y, color=color, linestyle="--", linewidth=1.5, zorder=2)
        # hollow markers only on the over-budget points (60-100)
        ax.plot([c["cost"] for c in over], [c["accuracy"] for c in over],
                color=color, marker=marker, markerfacecolor="none",
                linestyle="None", markersize=6,
                markeredgewidth=1.2, markeredgecolor=color, zorder=2)
# Overlay your own search curve (model color, thick solid line) if it exists.
# Your verify points are intentionally NOT overlaid: on 6 problems they are not
# comparable to the cached 100-problem curves at the same budget granularity
# (the equal-budget search-vs-verify comparison lives in your own figure, Section 3).
if exp is not None:
    _mine_color = next((c for k, _, _, _, c in _MODELS if k == MODEL), "#2E75B6")
    _mine = [{"cost": c["config"]["N"] * (1 + c["config"]["M"]), "acc": c["accuracy"],
              "search": c["config"]["strategy"] == "search"} for c in exp["configs"]]
    _mine_s = sorted([m for m in _mine if m["search"]], key=lambda m: m["cost"])
    _mine_label = next((label for k, label, _, _, _ in _MODELS if k == MODEL), MODEL)
    if _mine_s:
        ln, = ax.plot([m["cost"] for m in _mine_s], [m["acc"] for m in _mine_s],
                      color=_mine_color, marker="o", linestyle="solid",
                      linewidth=3.2, markersize=7.5, markeredgewidth=0.8,
                      markeredgecolor="#FFFFFF", label=_mine_label + " (yours, search)", zorder=4)
        _handles.append(ln)
ax.set_xlabel("Inference calls (budget)", fontsize=12)
ax.set_ylabel("Accuracy (MATH-500 subset)", fontsize=12)
ax.grid(True, axis="y", color="#E3E3E3", linewidth=0.7, zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#999999")
ax.spines["bottom"].set_color("#999999")
ax.tick_params(labelsize=11, colors="#333333")
ax.set_ylim(0.30, 0.92)
fig.subplots_adjust(top=0.80)
from matplotlib.patches import FancyBboxPatch
# Main legend: 8 entries in 2 rows x 4 cols, no frame (box drawn by hand below)
_leg1 = ax.legend(handles=_handles, loc="upper center", bbox_to_anchor=(0.5, 1.28), ncol=4,
                  fontsize=10, frameon=False, borderpad=0.6, columnspacing=1.0, handletextpad=0.5)
# Note line (3rd row, single line, inside the same box) + enclosing box.
# Two draws first: the first loads fonts, the second gives stable layout metrics.
# Everything is measured in display pixels, then converted to axes fractions, so the
# note aligns symbol-to-symbol and text-to-text with the FIRST legend entry.
from matplotlib import font_manager as _mpl_fm
fig.canvas.draw()
fig.canvas.draw()
_renderer = fig.canvas.get_renderer()
_ax_tr = ax.transAxes.inverted()
_fb = _leg1._legend_handle_box.get_window_extent(_renderer)  # entries bbox (px)
_hb = _leg1.legend_handles[0].get_window_extent(_renderer)   # first handle (line+marker)
_tb = _leg1.get_texts()[0].get_window_extent(_renderer)     # first label text
_note_text = "hollow markers: over-budget reference points (3 main models, N=10, M=5-9, cost > 50)"
_note_w, _note_h, _note_d = _renderer.get_text_width_height_descent(
    _note_text, _mpl_fm.FontProperties(size=10), ismath=False)
_note_cx = (_hb.x0 + _hb.x1) / 2.0      # symbol center = first handle center
_note_tx = _tb.x0                        # text left = first entry's text left
_note_cy = _fb.y0 - 6.0 - _note_h / 2.0  # just below the entries (6 px gap; display y grows up)
# enclosing box: tightly wraps entries + note (6 px padding)
_pad = 6.0
_bx0 = min(_fb.x0, _note_tx) - _pad
_bx1 = max(_fb.x1, _note_tx + _note_w) + _pad
_by0 = _note_cy - _note_h / 2.0 - _pad
_by1 = _fb.y1 + _pad
_a = _ax_tr.transform((_bx0, _by0))
_b = _ax_tr.transform((_bx1, _by1))
ax.add_patch(FancyBboxPatch(
    (_a[0], _a[1]), _b[0] - _a[0], _b[1] - _a[1],
    boxstyle="round,pad=0.01", fc="#FFFFFF", ec="#CCCCCC", lw=0.9,
    transform=ax.transAxes, zorder=3, clip_on=False))
# note symbol + text (axes coords)
_na = _ax_tr.transform((_note_cx, _note_cy))
_nn = _ax_tr.transform((_note_tx, _note_cy))
ax.plot([_na[0]], [_na[1]], marker="o", markerfacecolor="none", color="#888888",
        linestyle="--", linewidth=1.5, markersize=6.5, markeredgewidth=1.2,
        transform=ax.transAxes, clip_on=False, zorder=6)
ax.text(_nn[0], _nn[1], _note_text, fontsize=10, color="#666666",
        ha="left", va="center", transform=ax.transAxes, zorder=6, clip_on=False)
plt.show()









---

### 3b.1 Reading the Curve Family Figure

The main message: **at equal budget, search beats weak self-verification**. The three main models' over-budget verify points (cost 60-100) sit ≈6-17 pp below their search lines at cost 50; your own search-vs-verify comparison lives in your Section 3 figure. More re-scoring does not raise the verify level (Section 3b). If the over-budget points stay ≈6 pp or more below the search line at 50, verify at 50 is no better: search wins. In-budget verify points are not plotted for any cached model, to keep the figure readable. If your model is not one of the three main ones, you can read its cached budget-50 verify values from `data/cache_subset/curves.json` (the configs with strategy="verify" under your model key).

Self-checks after you look at the figure:

- In your Section 3 figure, does your verify point at cost 50 fall below your search point at cost 50? If yes, you reproduced the course's conditional claim (weak verifier, B=50); if not, check the error-dispersion cell (Section 3c) before concluding: with 6 problems a single wrong answer moves the curve by ≈17 pp. Your search points may land on the same accuracy across budgets (expected on 6 problems; the cached Gemma-3-1B curve is flat too).
- Are the over-budget verify points trending up with more verification? In our data they fluctuate around the budget-50 level rather than rising: verification has diminishing returns once candidates are sampled.
- How does your model's search slope compare with Qwen3-4B's (p=0.73) and Gemma-3-1B's (p=0.37)? If you have fewer than two search budgets, use your model's cached search curve instead. Relate what you see to p: the guaranteed zone starts at p>1/2; below it, voting can still help (error dispersion) or stall. The slope you observe is your data, not a prediction.


---

## 3c. Error Dispersion Check (5 minutes)

Whether or not you reproduced the claim, check dispersion in your own data: voting can *hurt* when wrong answers are concentrated (Section 1 note). If your N=50 vote was correct on every problem, there are no wrong votes to inspect; that is still a result (likely for other students of the same model; many Qwen3-4B students will see it). The included statistics then load as the reference: for the problems where the N=50 vote picked a **wrong** answer, how often did the losing candidates agree on the same wrong answer? This is a rough measure of error concentration.

<details><summary>What to look for</summary>

If most wrong votes picked the same wrong answer, errors are concentrated and voting is fragile. If wrong-answer votes are scattered across many different wrong answers, voting mostly failed by chance; no single wrong answer dominated: the "error dispersion" effect. This analysis is most informative in the no-guarantee zone (p<=1/2): no mathematical guarantee, but dispersion decides whether voting still helps.

</details>


In [ ]:
# Error dispersion check: majority-vote winners that were wrong - how concentrated?
import collections, json, os

disp = None
rec_path = os.path.join(OUT_DIR, "records.jsonl")
wrong_votes = []
if os.path.exists(rec_path):
    for line in open(rec_path, encoding="utf-8"):
        r = json.loads(line)
        if r.get("call_type") == "aggregate" and r.get("strategy") == "search"                 and r.get("N") == 50 and r.get("correct") is False:
            # votes = the winning answer's vote count (int); N = number of candidates.
            # share = winning share of all votes: close to 1.0 means the errors
            # converged on one wrong answer (concentrated); low share means the
            # losing candidates scattered across many different wrong answers.
            v = r.get("votes")
            n = r.get("N") or 50
            if v is not None:
                wrong_votes.append(v / max(1, n))
if not wrong_votes:
    # Cache fallback: use the included 100-problem dispersion statistics
    # (data/cache_subset/dispersion.json) so cache-layer students can do this
    # check too - with a statistically meaningful sample.
    disp_path = os.path.join("data", "cache_subset", "dispersion.json")
    if os.path.exists(disp_path):
        for e in json.load(open(disp_path, encoding="utf-8"))["dispersion"]:
            if e.get("key") == MODEL and e["dataset"] == "MATH-500":
                print("Using the included 100-problem statistics for", MODEL, "(no local records):")
                print(f"  Problems where the N=50 vote was wrong: {e['n_wrong_votes']}")
                print(f"  Average share of votes held by the (wrong) winning answer: {e['avg_winning_share']:.2f}")
                print("  A share close to 1.0 means errors are concentrated; scattered winners mean dispersed errors.")
                disp = e
                break
if disp is None and not wrong_votes:
    print("Note: no records yet and no cached dispersion data - run the experiment cell first, then re-run this cell.")
elif wrong_votes:
    avg_share = sum(wrong_votes) / len(wrong_votes)
    print(f"Problems where the N=50 vote was wrong: {len(wrong_votes)}")
    print(f"Average share of votes held by the (wrong) winning answer: {avg_share:.2f}")
    print("A share close to 1.0 means errors are concentrated; scattered winners mean dispersed errors.")
    if len(wrong_votes) < 3:
        print("(Small sample: with only 6 problems the count is indicative at best - compare with the cached statistics above.)")

---

## 4. Analysis Questions (Answer in your own words)

1. **Where is the crossing point?** Do the search-only configs (M=0) and verify-heavy configs (large M) cross within your measured budget range? Is the best point at high N (search) or high M (verify)?

   If you did not run the full grid, rely on your budget-50 verify points from your Section 3 figure, and on the over-budget references if your model is one of the three main ones. On the cache layer, read the budget-50 verify values from `data/cache_subset/curves.json` (strategy="verify"), as in Section 3b.1.

2. **How much budget would the verify strategy need to catch up with search at N=50?** In your Section 3 figure, verify N=5/M=9 uses 50 calls and lands at accuracy *a*; search N=50 uses 50 calls and lands at *b*. The cached over-budget points stay within ≈6 pp of the budget-50 verify level, ≈6-17 pp below the search lines, and do not rise with more re-scoring. Extrapolation: with the N=5 series (M=1 at 10 calls, M=9 at 50 calls), judge how far the verify trend would need to go to reach *b*; the verifier-quality ceiling explains why it does not. On the cache layer, as in Question 1.

   A weak verifier barely distinguishes right from wrong: measured from Qwen3-0.6B self-scoring runs (100-problem seed-0, scores on a 1-10 scale), alpha ≈ 0.62 = the fraction of wrong candidates scored >= 7, and beta ≈ 0.37 = the fraction of correct candidates scored < 7. The direction is robust, though the exact values depend on the counting rule. Why does this *verifier-quality ceiling* make the gap hard to close no matter how large M gets? (Hint: more re-scoring resamples the same weak signal; it does not get sharper with M.)

3. **p-spectrum interpretation:** locate your model's p in the table above. Is your model in the guaranteed zone (p>1/2) or the no-guarantee zone (p<=1/2)? If your model is in the no-guarantee zone: does your curve show voting helping? What does that imply about *error dispersion* vs. the mathematical guarantee? (In the guaranteed zone the guarantee already explains the outcome.)

4. **Class collaboration:** share your model, p, and curve with the class. Together you reconstruct the six-model p-spectrum. Only pool data from the **same shared 6-problem subset** (data comparability rule); if you are on the cache layer, share the cached curve labeled as the 100-problem reference instead; do not mix it into the 6-problem pool.

<details><summary>Discussion points (check after you answer)</summary>

1. In our headline comparison (3-core configs x 3 seeds) the best point sits at the search end (N=50/M=0) for all three main models, including the no-guarantee zone: the p>1/2 guarantee is sufficient, not necessary; dispersion decides. (On the full config grid, the p=0.37 model's search curve is roughly flat, ≈7 pp across budgets; "best" there is within noise.)
2. Extrapolating the verify trend: each additional re-scoring resamples the same weak signal, so even large M cannot close the gap. The verifier-quality ceiling, not the number of checks M, limits verification.
3. If your model is in the no-guarantee zone and voting still helped, that is the error-dispersion effect rather than a contradiction of Chernoff; check the error-dispersion cell (Section 3c) for supporting evidence.

</details>

> **Common misconception alert:** "more verification is always more accurate". Watch whether the verify-heavy split (N=5/M=9) beats the verify-light split (N=10/M=4) at the same 50-call budget: in our 100-problem data the two splits land within noise of each other and the winner flips by model (-6 to +5 pp). The reliable lesson is that more re-scoring does not keep paying off, as the over-budget points show.


---

## 5. (Optional) Weak vs. Strong Verifier Sub-Experiment

Run the same two configs (N=5/M=9 and N=10/M=4) twice: once with **self-scoring** (the generator model grades its own answers; a weak verifier) and once with **unified scoring** (Qwen3-4B grades all models; a relatively stronger verifier). The two modes share the same candidate pool (per-config seeds, judge-independent), isolating the verifier-quality effect; the calls and time below apply per mode.

This runs the first 30 problems of the shared 100-problem subset (2 configs x 30 problems). N=5/M=9 and N=10/M=4 each consume 50 calls per problem, so ≈3,000 calls per scoring mode; the self + unified pair totals ≈6,000 calls, a few hours per mode on a fast small model. Scale by your own throughput: calls x 700 tokens / tok/s (formula in Section 1). Its results answer an independent sub-question; do not pool them with your main 6-problem run (data comparability rule).

<details><summary>Cache-layer results</summary>

If you are on the cache layer, the cached results are in `data/cache_subset/unified.json`. It covers a different 30-problem set than your run: the two overlap by 24 problems, so treat the comparison as directional. It includes Gemma-3-1B in both modes and Qwen3-1.7B in unified mode only; for other models, or for a 1.7B self-scoring baseline, reason qualitatively.

</details>

**Note: this sub-experiment loads Qwen3-4B as the unified judge (≈5.3 GB) on top of your model, roughly 10-15 GB total VRAM (file + KV + process overhead per loaded model; 4B-class pairs need ≈14-15 GB). Skip it if your GPU cannot fit both.** If `MODEL` is Qwen3-4B itself, unified scoring coincides with self-scoring (skip the comparison).

**After it finishes, compare the two scoring settings:** does the unified (stronger) judge raise verify accuracy relative to self-scoring? Does the gain shrink as p increases? Does a stronger verifier change the winner at equal budget?


In [ ]:
# Optional: unified-verifier sub-experiment (~3-5 h per scoring mode on a small model; skip if short on time)
cmd = [sys.executable, "scripts/run_experiment.py",
       "--model", MODEL, "--seeds", "0", "--judge", "unified",
       "--configs", "N=5,M=9", "N=10,M=4",
       "--max-problems", "30", "--out-dir", "data/results/judge_unified_" + MODEL, "--finalize"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Unified-verifier sub-experiment finished.")


---

## Summary

- If you ran your own experiment, you have produced at least **2 data points** (search vs. verify at equal budget) for your model.
- The cached curve family (six models) lets you read the full p-spectrum even if your own run covered just a few configs.
- Key takeaway to test: under a weak verifier and moderate budget, **generating more candidates (search) beats scrutinizing fewer (verify)**. The p>1/2 voting guarantee explains why search works in the guaranteed zone; error dispersion does the rest for weaker models. The verifier-quality ceiling (re-scoring resamples the same weak signal) explains why the gap is hard to close.

Proceed to `02_optimal_ratio.ipynb` when ready.
